# Dtypes, PM plots and exposure

This example demonstrates how to:

- Load size-resolved ELPI data (`Aerosol2D`)
- Inspect dtype, units, and particle density
- Convert between different distribution types via `dtype_converter`
- Compute cumulative Pₓ (e.g. PM₂.₅, PM₄.₂) with `pm_calc`
- Plot PMₓ time series with `plot_PM_timeseries`
- Adjust assumed particle density with `set_density`
- Use `summarize_activities` and `summarize_exposure` for PM-based metrics

In [ ]:
import matplotlib.pyplot as plt
import aerosoltools as at

## Load ELPI sample data

Initially load an ELPI test dataset.

In [ ]:
filename = "../../tests/data/Sample_ELPI.txt"
elpi = at.load_elpi_file(filename)

A warning is given, seeing that the density in the given raw data is not set equal to 1.0. The density of the raw data is read, loaded and stored within the generated elpi variable, but the user is still alerted.

## Check dtype, unit, and density

The loader sets the initial distribution type (e.g. `dN`) and unit.
Assumed particle density is stored in `elpi.density` (g/cm³).

In [ ]:
print("Initial dtype:", elpi.dtype)
print("Initial unit:", elpi.unit)
print("Initial density (g/cm³):", elpi.density)

## Change density and convert to mass distribution

We can update the assumed particle density with `set_density()` and then
convert to a mass-based size distribution using `dtype_converter("dM")`.

In [ ]:
# For illustration, set density to 1.5 g/cm³
elpi.set_density(1.5)
elpi.dtype_converter("dM")
print("Updated dtype:", elpi.dtype)
print("Updated density (g/cm³):", elpi.density)
print("Updated unit:", elpi.unit)

## Compute PM fractions with `pm_calc`

We now calculate mass-based Pₓ metrics such as PM₂.₅ and PM₄.₂.
The results are stored in `elpi.extra_data` as additional columns.

In [ ]:
# Compute cumulative PM2.5 and PM4.2 (µm cut points)
elpi.pm_calc(dtype="dM", PM=2.5)
elpi.pm_calc(dtype="dM", PM=4.2)
print([c for c in elpi.extra_data.columns if c.startswith("PM")])


## Plot PM time series

Use `plot_PM_timeseries` to visualize the evolution of PMₓ over time.

In [ ]:
elpi.plot_PM_timeseries(PM_values=[2.5, 4.2, 10])

## Define activities and summarize

As with `Aerosol1D`, we can mark time segments and compute activity-based
statistics using `summarize_activities()`.

In [ ]:
activity_periods = {
    "Background": [("2023/09/07 09:06:50", "2023/09/07 09:07:50")],
    "Emission":   [("2023/09/07 09:07:55", "2023/09/07 09:08:30")],
    "Decay":      [("2023/09/07 09:09:00", "2023/09/07 09:10:50")],
}
elpi.mark_activities(activity_periods)
summary = elpi.summarize_activities()
summary

## Exposure summary for respirable dust (PM4.2)

Finally, we use `summarize_exposure` on the `Aerosol2D` object
to obtain exposure metrics for PM₄.₂ during the *Emission* task,
using *Background* as the background activity.

In [ ]:
exp = elpi.summarize_exposure(
    metric="PM4.2",
    activities=["Emission"],
    background="Background",  # use mean Background as background level
    exposure_hours=None,       # assume exposure equals task duration
    short_limit=1.0,
    long_limit=1.0,
    short_window="15min",
    twa_window="8h",
)
exp